# 📞 Task 5: Personal Loan Acceptance Prediction
**Internship: DevelopersHub Corporation — Data Science & Analytics**

---
## 1. Introduction & Problem Statement
Predict which bank marketing campaign customers are likely to accept a personal loan/term deposit offer.

**Dataset:** Bank Marketing Dataset (UCI)  
**Model:** Decision Tree Classifier

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
sns.set_theme(style='whitegrid')
print('✅ Libraries imported')

## 2. Dataset Understanding

In [ ]:
df = pd.read_csv('bank_marketing.csv')
print('Shape:', df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(16,5))
sns.histplot(data=df, x='age', hue='y', kde=True, ax=axes[0])
axes[0].set_title('Age Distribution by Acceptance')

top_jobs = df.groupby('job')['y'].apply(lambda s: (s=='yes').mean()).sort_values(ascending=False)
top_jobs.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Acceptance Rate by Job')
axes[1].tick_params(axis='x', rotation=75)

df.groupby('marital')['y'].apply(lambda s: (s=='yes').mean()).plot(kind='bar', ax=axes[2], color='coral')
axes[2].set_title('Acceptance Rate by Marital Status')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150)
plt.show()

## 4. Data Preparation

In [ ]:
df_enc = df.copy()
le = LabelEncoder()
cat_cols = ['job','marital','education','default','housing','loan','contact','month','poutcome']
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col])
df_enc['y'] = df_enc['y'].map({'yes':1,'no':0})

X = df_enc.drop(columns=['y'])
y = df_enc['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('✅ Data encoded and split')

## 5. Model Training

In [ ]:
model = DecisionTreeClassifier(max_depth=6, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('✅ Model trained')

## 6. Evaluation

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f}')
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['No','Yes'], cmap='Greens')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 7. Business Insight Extraction

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8,5))
importances.head(10).plot(kind='barh', color='teal')
plt.title('Top 10 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print(importances.head(10))

## 8. Conclusion
- Call **duration** and previous campaign **outcome (poutcome)** are the strongest predictors of acceptance.
- Customers without existing housing/personal loans are more likely to accept new offers.
- Business takeaway: prioritize re-contacting customers with prior successful campaign outcomes and longer engagement potential.